> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.14 - ORM

## Exercícios

#### Q1. Conhecendo os dados
Baixe o seguinte csv onde iremos trabalhar. Ele contém informações sobre salários de profissionais de dados de uma empresa hipotética entre 2009 e 2016
* https://github.com/camilalaranjeira/python-intermediario/blob/main/salaries.csv

Suas colunas, descritas na [página do Kaggle que contém o dataset](https://www.kaggle.com/datasets/krishujeniya/salary-prediction-of-data-professions?resource=download), são:
* FIRST NAME: Primeiro nome do profissional de dados (String)
* LAST NAME: Sobrenome do profissional de dados (String)
* SEX: Gênero do profissional de dados (String: 'F' para Feminino, 'M' para Masculino)
* DOJ (Date of Joining): A data em que o profissional de dados ingressou na empresa (Data no formato MM/DD/AAAA)
* CURRENT DATE: A data atual ou a data de referência dos dados (Data no formato MM/DD/AAAA)
* DESIGNATION: O cargo ou designação do profissional de dados (String: ex., Analista, Analista Sênior, Gerente)
* AGE: Idade do profissional de dados (Integer)
* SALARY: Salário anual do profissional de dados (Float)
* UNIT: Unidade de negócios ou departamento em que o profissional de dados trabalha (String: ex., TI, Finanças, Marketing)
* LEAVES USED: Número de licenças utilizadas pelo profissional de dados (Integer)
* LEAVES REMAINING: Número de licenças restantes para o profissional de dados (Integer)
* RATINGS: Avaliações de desempenho do profissional de dados (Float)
* PAST EXP: Experiência de trabalho anterior em anos antes de ingressar na empresa atual (Float)

Na célula a seguir, **carregue os dados do CSV e dê uma olhada neles antes de seguir**.

In [2]:
import pandas as pd

df = pd.read_csv("salaries.csv")

print(df.head())

  FIRST NAME   LAST NAME SEX         DOJ CURRENT DATE DESIGNATION   AGE  \
0     TOMASA       ARMEN   F   5-18-2014   01-07-2016     Analyst  21.0   
1      ANNIE         NaN   F         NaN   01-07-2016   Associate   NaN   
2      OLIVE        ANCY   F   7-28-2014   01-07-2016     Analyst  21.0   
3     CHERRY     AQUILAR   F  04-03-2013   01-07-2016     Analyst  22.0   
4       LEON  ABOULAHOUD   M  11-20-2014   01-07-2016     Analyst   NaN   

   SALARY        UNIT  LEAVES USED  LEAVES REMAINING  RATINGS  PAST EXP  
0   44570     Finance         24.0               6.0      2.0         0  
1   89207         Web          NaN              13.0      NaN         7  
2   40955     Finance         23.0               7.0      3.0         0  
3   45550          IT         22.0               8.0      3.0         0  
4   43161  Operations         27.0               3.0      NaN         3  


#### Q2. Modelando os dados
Você deve **criar um ORM com SQLAlchemy capaz de comportar os dados dessa base**.

* Crie um campo de chave primária `ID`, que deve ser incrementado automaticamente
* Os campos SEX, DESIGNATION e UNIT devem ser definidos como classes `Enum` com os possíveis valores (consulte os valores únicos dessas colunas)
* Para os outros campos, consulte os tipos de dados informados na descrição acima

In [ ]:
import enum

class SexEnum(enum.Enum):
    F = "F"
    M = "M"


class DesignationEnum(enum.Enum):
    Analyst = "Analyst"
    Associate = "Associate"
    # adicione outros conforme aparecerem no dataset


class UnitEnum(enum.Enum):
    Finance = "Finance"
    Web = "Web"
    IT = "IT"
    Operations = "Operations"
    # complete conforme necessário

In [ ]:
from sqlalchemy import (
    Column, Integer, String, Float, Date, Enum
)
from sqlalchemy.orm import declarative_base

Base = declarative_base()


class Employee(Base):
    __tablename__ = "employees"

    id = Column(Integer, primary_key=True, autoincrement=True)

    first_name = Column(String, nullable=False)
    last_name = Column(String, nullable=True)

    sex = Column(Enum(SexEnum), nullable=True)

    doj = Column(Date, nullable=True)  # Date of Joining
    current_date = Column(Date, nullable=False)

    designation = Column(Enum(DesignationEnum), nullable=True)

    age = Column(Float, nullable=True)

    salary = Column(Integer, nullable=False)

    unit = Column(Enum(UnitEnum), nullable=True)

    leaves_used = Column(Float, nullable=True)
    leaves_remaining = Column(Float, nullable=True)

    ratings = Column(Float, nullable=True)

    past_exp = Column(Integer, nullable=True)

In [ ]:
df["DOJ"] = pd.to_datetime(df["DOJ"], errors="coerce")
df["CURRENT DATE"] = pd.to_datetime(df["CURRENT DATE"], errors="coerce")

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///salaries.db")

Base.metadata.create_all(engine)

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)
session = Session()

for _, row in df.iterrows():
    emp = Employee(
        first_name=row["FIRST NAME"],
        last_name=row["LAST NAME"],
        sex=row["SEX"] if pd.notna(row["SEX"]) else None,
        doj=row["DOJ"],
        current_date=row["CURRENT DATE"],
        designation=row["DESIGNATION"],
        age=row["AGE"],
        salary=int(row["SALARY"]),
        unit=row["UNIT"],
        leaves_used=row["LEAVES USED"],
        leaves_remaining=row["LEAVES REMAINING"],
        ratings=row["RATINGS"],
        past_exp=row["PAST EXP"]
    )
    session.add(emp)

session.commit()

In [ ]:
sex = SexEnum(row["SEX"]) if pd.notna(row["SEX"]) else None

#### Q3. Estabelecendo uma conexão

Usando o método `create_engine` do SQLAlchemy, crie uma conexão com um novo banco de dados SQLite chamado `salarios`.

In [4]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///salarios.db")

#### Q4. Criando as tabelas
Crie as tabelas da questão Q2 no banco `salarios`.

In [ ]:
from sqlalchemy import create_engine

# 1. Criar engine
engine = create_engine("sqlite:///salarios.db")

# 2. Criar todas as tabelas mapeadas no Base
Base.metadata.create_all(engine)

#### Q5. Populando

Usando o método `to_sql` da biblioteca Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)), popule o banco `salarios` com os dados do csv que você carregou na questão Q1.
* Lembre-se de definir o parâmetro `if_exists='append'` para que as tabelas não sejam dropadas e recriadas.

In [ ]:
df.columns = [
    "first_name", "last_name", "sex", "doj", "current_date",
    "designation", "age", "salary", "unit",
    "leaves_used", "leaves_remaining", "ratings", "past_exp"
]

In [ ]:
df["doj"] = pd.to_datetime(df["doj"], errors="coerce")
df["current_date"] = pd.to_datetime(df["current_date"], errors="coerce")

In [ ]:
if "id" in df.columns:
    df = df.drop(columns=["id"])

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///salarios.db")

df.to_sql(
    name="employees",
    con=engine,
    if_exists="append",
    index=False
)

#### Q6. Consultas SQL vs ORM

Agrupe os dados por DESIGNATION e selecione o mínimo, máximo e a média dos salários (SALARY) divididos por 12. Já que o atributo SALARY é anual, dividir por 12 nos mostrará os valores mensais.

Assumindo que a variável que armazena a sua conexão se chama `engine`, você deve realizar a query acima de três formas:
* Executando a query SQL através de uma instância de conexão retornada pelo método `engine.connect()`
* Executando a query SQL com o método `read_sql_query` do Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)). Você usará mesma instância `engine.connect()` como um dos parâmetros.
* Executando uma query criada com o módulo `select` do SQLAlchemy. Sua execução deve ser feita através de um objeto `Session` do módulo `orm` do SQLAlchemy (`Session(engine)`).


In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///salarios.db")

query = text("""
SELECT 
    designation,
    MIN(salary)/12 AS min_salary,
    MAX(salary)/12 AS max_salary,
    AVG(salary)/12 AS avg_salary
FROM employees
GROUP BY designation
""")

with engine.connect() as conn:
    result = conn.execute(query)

    for row in result:
        print(row)

In [ ]:
import pandas as pd

with engine.connect() as conn:
    df_result = pd.read_sql_query("""
        SELECT 
            designation,
            MIN(salary)/12 AS min_salary,
            MAX(salary)/12 AS max_salary,
            AVG(salary)/12 AS avg_salary
        FROM employees
        GROUP BY designation
    """, conn)

print(df_result)

In [ ]:
from sqlalchemy import select, func
from sqlalchemy.orm import Session

with Session(engine) as session:
    stmt = select(
        Employee.designation,
        (func.min(Employee.salary) / 12).label("min_salary"),
        (func.max(Employee.salary) / 12).label("max_salary"),
        (func.avg(Employee.salary) / 12).label("avg_salary"),
    ).group_by(Employee.designation)

    result = session.execute(stmt)

    for row in result:
        print(row)